# NQ Session Direction Predictor

**Goal:** Identify market context variables — observable *before* a session starts — that predict session direction.

Two sessions are analysed:
- **Initial Balance (IB):** 9:30–10:30am NY
- **NY Regular Session:** 9:30am–4:00pm NY

### Workflow
| Cell | Purpose |
|------|---------|
| **Cell 1** | Configuration — set all paths and parameters here |
| **Cell 2** | Build dataset — loads raw 1-min data, engineers features, saves `session_dataset.csv` |
| **Cell 3** | Combinatorial analysis — tests 1/2/3/4-feature combos for both sessions, saves CSVs |
| **Cell 4** | Display results — shows top tradeable patterns with actionability scores |

**Run cells in order.** Cell 3 and Cell 4 load from disk, so Cell 2 must be run first.

---
## Cell 1 — Configuration
**Edit this cell to match your environment.** All other cells read from these variables.

In [1]:
import os

# ============================================================
# PATHS — update these to match your machine
# ============================================================

# Folder containing nq_15y_1min.csv
DATA_DIR = r"C:\Users\tammm\NQ and ES analysis"

# Raw 1-minute data file
RAW_DATA_FILE = "nq_15y_1min.csv"

# Where to save the built dataset and analysis CSVs
OUTPUT_DIR = r"C:\Users\tammm\NQ and ES analysis"

# ============================================================
# DATE RANGE
# ============================================================

START_DATE = "2013-01-01"
END_DATE   = "2025-12-31"

# ============================================================
# ANALYSIS PARAMETERS
# ============================================================

# Minimum sample sizes per combo level
MIN_SAMPLES = {
    1: 30,   # individual features
    2: 25,   # 2-feature combos
    3: 20,   # 3-feature combos
    4: 15,   # 4-feature combos
}

# Minimum edge % from 50/50 to include a result
MIN_EDGE_PCT = 5.0

# Quality score weighting
EDGE_WEIGHT   = 0.6
SAMPLE_WEIGHT = 0.4

# Tradeable pattern filters (Cell 4)
TRADEABLE_MIN_SAMPLE     = 50
TRADEABLE_MIN_CONFIDENCE = 65.0  # %

# How many top features to use for 3-feature combos
TOP_N_FOR_3FEATURE = 10
# How many top features to use for 4-feature combos
TOP_N_FOR_4FEATURE = 8

# ============================================================
# DERIVED PATHS — no need to edit below
# ============================================================

RAW_DATA_PATH   = os.path.join(DATA_DIR, RAW_DATA_FILE)
DATASET_PATH    = os.path.join(OUTPUT_DIR, "session_dataset.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration loaded.")
print(f"  Raw data : {RAW_DATA_PATH}")
print(f"  Dataset  : {DATASET_PATH}")
print(f"  Outputs  : {OUTPUT_DIR}")

Configuration loaded.
  Raw data : C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv
  Dataset  : C:\Users\tammm\NQ and ES analysis\session_dataset.csv
  Outputs  : C:\Users\tammm\NQ and ES analysis


---
## Cell 2 — Build Dataset

Loads raw 1-min NQ data and engineers one row per trading day containing:
- **Targets:** IB direction (9:30–10:30am) and NY session direction (9:30am–4:00pm)
- **Pre-IB features:** Everything observable *before* 9:30am — sessions, ORBs, pivots, gap, positional relationships
- **IB structure features** *(NY session predictor only):* `ib_close_half` (upper/lower half of IB range) and `ib_extreme_first` (whether the IB high or low printed first)

Output saved to `session_dataset.csv`.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Pivot helpers ────────────────────────────────────────────────────────────

def calculate_floor_pivots(high, low, close):
    """Calculate classic floor pivot levels plus golden zone."""
    pp = (high + low + close) / 3
    r1 = (2 * pp) - low
    r2 = pp + (high - low)
    r3 = high + 2 * (pp - low)
    s1 = (2 * pp) - high
    s2 = pp - (high - low)
    s3 = low - 2 * (high - pp)
    pivot_range = high - low
    gz_low  = low + pivot_range * 0.500
    gz_high = low + pivot_range * 0.618
    return dict(PP=pp, R1=r1, R2=r2, R3=r3, S1=s1, S2=s2, S3=s3,
                GZ_LOW=gz_low, GZ_HIGH=gz_high)


def get_pivot_position(price, pivots, threshold_pct=0.1):
    """Return which pivot zone a price sits in."""
    thr = pivots['PP'] * (threshold_pct / 100)
    if abs(price - pivots['PP']) <= thr:          return 'at_pp'
    if pivots['GZ_LOW'] <= price <= pivots['GZ_HIGH']: return 'in_gz'
    if price >= pivots['R3']:                     return 'above_r3'
    if price >= pivots['R2']:                     return 'r2_to_r3'
    if price >= pivots['R1']:                     return 'r1_to_r2'
    if price > pivots['GZ_HIGH']:                 return 'gz_to_r1'
    if price > pivots['PP']:                      return 'pp_to_gz'
    if price <= pivots['S3']:                     return 'below_s3'
    if price <= pivots['S2']:                     return 's3_to_s2'
    if price <= pivots['S1']:                     return 's2_to_s1'
    if price < pivots['GZ_LOW']:                  return 's1_to_gz'
    return 'gz_to_pp'


def level_touched(data, level_price, threshold_pct=0.05):
    """Return True if any bar in `data` came within threshold of level_price."""
    if data.empty or level_price is None:
        return False
    thr = level_price * (threshold_pct / 100)
    return bool(((data['Low'] - thr) <= level_price).any() and
                ((data['High'] + thr) >= level_price).any())


def orb_direction(data, min_bars=5):
    """Return 'bullish'/'bearish'/None for an ORB period."""
    if data.empty or len(data) < min_bars:
        return None
    o, c = data['Open'].iloc[0], data['Close'].iloc[-1]
    return 'bullish' if c > o else ('bearish' if c < o else None)


def session_direction(data):
    """Return 'bullish'/'bearish'/None for any session."""
    if data.empty:
        return None
    o, c = data['Open'].iloc[0], data['Close'].iloc[-1]
    if o == c:
        return None
    return 'bullish' if c > o else 'bearish'


# ── Load raw data ────────────────────────────────────────────────────────────

print("Loading raw data...")
df = pd.read_csv(RAW_DATA_PATH)
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)
df = df.loc[START_DATE:END_DATE]

# Group by NY trading day (day boundary at 6pm)
df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()
dates = df['NY_Trading_Day'].unique()

print(f"Loaded {len(df):,} bars | {len(dates)} trading days ({START_DATE} → {END_DATE})")
print()

# ── Build dataset ────────────────────────────────────────────────────────────

PIVOT_ORDER = ['R3', 'R2', 'R1', 'PP', 'S1', 'S2', 'S3']

records  = []
skipped  = 0

print("Building dataset...")
for idx, date in enumerate(dates[1:], 1):
    if idx % 500 == 0:
        print(f"  Day {idx} of {len(dates)-1}...")

    actual = date + pd.Timedelta(hours=7)

    # Full NY trading day: 6pm prior day → 5pm today
    day_start = actual.replace(hour=18) - pd.Timedelta(days=1)
    day_end   = actual.replace(hour=17)
    full_day  = df.loc[day_start:day_end]
    if full_day.empty:
        skipped += 1; continue

    # ── Targets ──────────────────────────────────────────────────────────────

    ib_start  = actual.replace(hour=9,  minute=30)
    ib_end    = actual.replace(hour=10, minute=30)
    ny_end    = actual.replace(hour=16, minute=0)

    ib_data = df.loc[ib_start:ib_end]
    ny_data = df.loc[ib_start:ny_end]   # NY session = 9:30am → 4:00pm

    if ib_data.empty or ny_data.empty:
        skipped += 1; continue

    ib_open, ib_close = ib_data['Open'].iloc[0], ib_data['Close'].iloc[-1]
    if pd.isna([ib_open, ib_close]).any() or ib_open == ib_close:
        skipped += 1; continue

    ny_open, ny_close = ny_data['Open'].iloc[0], ny_data['Close'].iloc[-1]
    if pd.isna([ny_open, ny_close]).any() or ny_open == ny_close:
        skipped += 1; continue

    ib_direction_target = 'bullish' if ib_close > ib_open else 'bearish'
    ny_direction_target = 'bullish' if ny_close > ny_open else 'bearish'

    ib_high  = ib_data['High'].max()
    ib_low   = ib_data['Low'].min()
    ib_range = ib_high - ib_low
    ny_range = ny_data['High'].max() - ny_data['Low'].min()

    # ── IB close position (upper / lower half of IB range) ───────────────────

    ib_midpoint  = (ib_high + ib_low) / 2
    ib_close_half = 'upper' if ib_close >= ib_midpoint else 'lower'

    # ── IB extreme formed first (high or low printed first within the IB) ────

    ib_extreme_first = None
    if not ib_data.empty and ib_range > 0:
        for _, bar in ib_data.iterrows():
            hit_high = bar['High'] >= ib_high
            hit_low  = bar['Low']  <= ib_low
            if hit_high and hit_low:
                # Both extremes in the same bar — call it simultaneous
                ib_extreme_first = 'simultaneous'
                break
            elif hit_high:
                ib_extreme_first = 'high'
                break
            elif hit_low:
                ib_extreme_first = 'low'
                break

    # ── Previous day ─────────────────────────────────────────────────────────

    prev_actual = (date - pd.Timedelta(days=1)) + pd.Timedelta(hours=7)
    prev_data   = df.loc[prev_actual.replace(hour=18) - pd.Timedelta(days=1)
                         : prev_actual.replace(hour=17)]

    if not prev_data.empty:
        prev_high  = prev_data['High'].max()
        prev_low   = prev_data['Low'].min()
        prev_open  = prev_data['Open'].iloc[0]
        prev_close = prev_data['Close'].iloc[-1]
        prev_range = prev_high - prev_low
        prev_direction = 'bullish' if prev_close > prev_open else 'bearish'
    else:
        prev_high = prev_low = prev_open = prev_close = prev_range = None
        prev_direction = None

    # ── Previous week ─────────────────────────────────────────────────────────

    cur_idx = list(dates).index(date)
    if cur_idx >= 5:
        wk_frames = []
        for wd in dates[cur_idx-5:cur_idx]:
            wa = wd + pd.Timedelta(hours=7)
            wdf = df.loc[wa.replace(hour=18) - pd.Timedelta(days=1): wa.replace(hour=17)]
            if not wdf.empty:
                wk_frames.append(wdf)
        if wk_frames:
            wk = pd.concat(wk_frames)
            prev_week_high = wk['High'].max()
            prev_week_low  = wk['Low'].min()
        else:
            prev_week_high = prev_week_low = None
    else:
        prev_week_high = prev_week_low = None

    # ── Floor pivots ──────────────────────────────────────────────────────────

    pivots = (calculate_floor_pivots(prev_high, prev_low, prev_close)
              if all(v is not None for v in [prev_high, prev_low, prev_close])
              else None)

    # ── Overnight (6pm → 9:30am) ──────────────────────────────────────────────

    on_data = df.loc[day_start: actual.replace(hour=9, minute=30)]

    on_dir = on_high = on_low = on_open = on_close = on_range = None
    gap_type = gap_size = None
    on_touch_prev_high = on_touch_prev_low = False
    on_touch_week_high = on_touch_week_low = False
    on_touch_r3 = on_touch_r2 = on_touch_r1 = False
    on_touch_pp = on_touch_s1 = on_touch_s2 = on_touch_s3 = on_touch_gz = False
    on_highest_pivot = on_lowest_pivot = None
    ny_open_pivot_pos = ny_open_vs_prev = None

    if not on_data.empty:
        on_open  = on_data['Open'].iloc[0]
        on_close = on_data['Close'].iloc[-1]
        on_high  = on_data['High'].max()
        on_low   = on_data['Low'].min()
        on_range = on_high - on_low
        on_dir   = 'bullish' if on_close > on_open else 'bearish'

        # Gap
        if prev_close and prev_range:
            gap_size = on_open - prev_close
            if   gap_size >  prev_range * 0.002: gap_type = 'gap_up'
            elif gap_size < -prev_range * 0.002: gap_type = 'gap_down'
            else:                                gap_type = 'no_gap'

        # Previous day / week level touches
        on_touch_prev_high = level_touched(on_data, prev_high)
        on_touch_prev_low  = level_touched(on_data, prev_low)
        on_touch_week_high = level_touched(on_data, prev_week_high)
        on_touch_week_low  = level_touched(on_data, prev_week_low)

        # Pivot touches
        if pivots:
            on_touch_r3 = level_touched(on_data, pivots['R3'])
            on_touch_r2 = level_touched(on_data, pivots['R2'])
            on_touch_r1 = level_touched(on_data, pivots['R1'])
            on_touch_pp = level_touched(on_data, pivots['PP'])
            on_touch_s1 = level_touched(on_data, pivots['S1'])
            on_touch_s2 = level_touched(on_data, pivots['S2'])
            on_touch_s3 = level_touched(on_data, pivots['S3'])
            on_touch_gz = (on_low <= pivots['GZ_HIGH'] and on_high >= pivots['GZ_LOW'])

            touched = [p for p in PIVOT_ORDER
                       if level_touched(on_data, pivots[p])]
            on_highest_pivot = touched[0]  if touched else None
            on_lowest_pivot  = touched[-1] if touched else None

            ny_open_pivot_pos = get_pivot_position(on_close, pivots)

        # NY open vs previous day range
        if on_close and prev_high and prev_low:
            if   on_close > prev_high: ny_open_vs_prev = 'above_prev_high'
            elif on_close < prev_low:  ny_open_vs_prev = 'below_prev_low'
            else:                      ny_open_vs_prev = 'inside_prev_range'

    # ── Asia session (6pm → 2am) ───────────────────────────────────────────────

    asia_data = df.loc[day_start: actual.replace(hour=2)]
    asia_dir  = session_direction(asia_data)
    asia_high = asia_data['High'].max()  if not asia_data.empty else None
    asia_low  = asia_data['Low'].min()   if not asia_data.empty else None
    asia_range= (asia_high - asia_low)   if asia_high else None

    # ── London session (2am → 8am) ─────────────────────────────────────────────

    lon_data  = df.loc[actual.replace(hour=2): actual.replace(hour=8)]
    lon_dir   = session_direction(lon_data)
    lon_high  = lon_data['High'].max()   if not lon_data.empty else None
    lon_low   = lon_data['Low'].min()    if not lon_data.empty else None
    lon_range = (lon_high - lon_low)     if lon_high else None

    # ── Asia/London positional relationship ────────────────────────────────────

    asia_lon_pos = None
    if all(v is not None for v in [asia_high, asia_low, lon_high, lon_low]):
        if   lon_low >= asia_low and lon_high <= asia_high:   asia_lon_pos = 'asia_engulfs_london'
        elif asia_low >= lon_low and asia_high <= lon_high:   asia_lon_pos = 'london_engulfs_asia'
        elif lon_high > asia_high and lon_low >= asia_low:    asia_lon_pos = 'london_partial_up'
        elif lon_low < asia_low  and lon_high <= asia_high:   asia_lon_pos = 'london_partial_down'
        elif lon_high > asia_high and lon_low < asia_low:     asia_lon_pos = 'london_breaks_both'
        else:                                                  asia_lon_pos = 'overlapping'

    # ── Session pattern (Asia + London combined direction) ─────────────────────

    session_pattern = None
    if asia_dir and lon_dir:
        session_pattern = f"{asia_dir[:4]}_{lon_dir[:4]}"  # e.g. bull_bear

    # ── ORBs ───────────────────────────────────────────────────────────────────

    # Globex ORB: first 30 min after 6pm
    globex_orb_dir = orb_direction(df.loc[day_start: day_start + pd.Timedelta(minutes=30)])

    # London ORB: 2am–2:30am
    lon_orb_dir = orb_direction(df.loc[actual.replace(hour=2): actual.replace(hour=2, minute=30)])

    # Pre-market ORB: 8am–9:30am
    pm_orb_dir = orb_direction(df.loc[actual.replace(hour=8): actual.replace(hour=9, minute=30)],
                               min_bars=1)

    # ── NY 5-min ORB (9:30–9:35am) ────────────────────────────────────────────

    orb5_start = actual.replace(hour=9, minute=30)
    orb5_end   = actual.replace(hour=9, minute=35)
    orb5_data  = df.loc[orb5_start:orb5_end]

    ny_orb_dir = ny_orb_first_break = ny_orb_range = None

    if not orb5_data.empty and len(orb5_data) >= 5:
        orb5_open  = orb5_data['Open'].iloc[0]
        orb5_close = orb5_data['Close'].iloc[-1]
        orb5_high  = orb5_data['High'].max()
        orb5_low   = orb5_data['Low'].min()
        ny_orb_range = orb5_high - orb5_low

        if   orb5_close > orb5_open: ny_orb_dir = 'bullish'
        elif orb5_close < orb5_open: ny_orb_dir = 'bearish'
        else:                        ny_orb_dir = 'flat'

        # Which extreme of the ORB broke first in the remainder of the IB?
        rest = df.loc[orb5_end: ib_end]
        if not rest.empty:
            hi_broken = lo_broken = False
            for _, row in rest.iterrows():
                if not hi_broken and row['High'] > orb5_high:
                    ny_orb_first_break = 'high'; hi_broken = True
                if not lo_broken and row['Low']  < orb5_low:
                    if ny_orb_first_break is None:
                        ny_orb_first_break = 'low'
                    lo_broken = True
                if hi_broken and lo_broken:
                    break
            if ny_orb_first_break is None:
                ny_orb_first_break = 'neither'

    # ── Day of week ────────────────────────────────────────────────────────────

    day_of_week = actual.strftime('%A')

    # ── Store record ───────────────────────────────────────────────────────────

    records.append({
        'date'        : actual.date(),
        'day_of_week' : day_of_week,

        # ── Targets ────────────────────────────────────────────────────────────
        'ib_direction': ib_direction_target,
        'ny_direction': ny_direction_target,

        # ── Range info (not used as predictors but useful context) ─────────────
        'ib_range'    : ib_range,
        'ny_range'    : ny_range,

        # ── IB structure features (observable by 10:30am, predict NY session) ──
        'ib_close_half'    : ib_close_half,
        'ib_extreme_first' : ib_extreme_first,

        # ── Previous day ───────────────────────────────────────────────────────
        'prev_day_direction' : prev_direction,
        'prev_day_range'     : prev_range,

        # ── Overnight ──────────────────────────────────────────────────────────
        'overnight_direction' : on_dir,
        'overnight_range'     : on_range,
        'gap_type'            : gap_type,
        'gap_size'            : gap_size,

        # ── Sessions ───────────────────────────────────────────────────────────
        'asia_direction'     : asia_dir,
        'london_direction'   : lon_dir,
        'globex_direction'   : on_dir,   # globex == overnight
        'session_pattern'    : session_pattern,
        'asia_london_position': asia_lon_pos,

        # ── ORBs ───────────────────────────────────────────────────────────────
        'globex_orb_direction'   : globex_orb_dir,
        'london_orb_direction'   : lon_orb_dir,
        'premarket_orb_direction': pm_orb_dir,
        'ny_orb_direction'       : ny_orb_dir,
        'ny_orb_first_break'     : ny_orb_first_break,
        'ny_orb_range'           : ny_orb_range,

        # ── Level touches (overnight) ─────────────────────────────────────────
        'overnight_touched_prev_day_high' : on_touch_prev_high,
        'overnight_touched_prev_day_low'  : on_touch_prev_low,
        'overnight_touched_prev_week_high': on_touch_week_high,
        'overnight_touched_prev_week_low' : on_touch_week_low,

        # ── Pivot touches (overnight) ─────────────────────────────────────────
        'overnight_touched_r3'     : on_touch_r3,
        'overnight_touched_r2'     : on_touch_r2,
        'overnight_touched_r1'     : on_touch_r1,
        'overnight_touched_pp'     : on_touch_pp,
        'overnight_touched_gz'     : on_touch_gz,
        'overnight_touched_s1'     : on_touch_s1,
        'overnight_touched_s2'     : on_touch_s2,
        'overnight_touched_s3'     : on_touch_s3,
        'overnight_highest_pivot'  : on_highest_pivot,
        'overnight_lowest_pivot'   : on_lowest_pivot,

        # ── NY open position ──────────────────────────────────────────────────
        'ny_open_pivot_position': ny_open_pivot_pos,
        'ny_open_vs_prev_day'   : ny_open_vs_prev,
    })

session_df = pd.DataFrame(records)

# Save
session_df.to_csv(DATASET_PATH, index=False)

print(f"\nDataset built: {len(session_df):,} trading days  |  Skipped: {skipped}")
print(f"IB  → Bullish: {(session_df['ib_direction']=='bullish').sum()}  "
      f"Bearish: {(session_df['ib_direction']=='bearish').sum()}")
print(f"NY  → Bullish: {(session_df['ny_direction']=='bullish').sum()}  "
      f"Bearish: {(session_df['ny_direction']=='bearish').sum()}")
print(f"Saved to: {DATASET_PATH}")

Loading raw data...
Loaded 4,161,727 bars | 3786 trading days (2013-01-01 → 2025-12-31)

Building dataset...
  Day 500 of 3785...
  Day 1000 of 3785...
  Day 1500 of 3785...
  Day 2000 of 3785...
  Day 2500 of 3785...
  Day 3000 of 3785...
  Day 3500 of 3785...

Dataset built: 3,109 trading days  |  Skipped: 676
IB  → Bullish: 1627  Bearish: 1482
NY  → Bullish: 1703  Bearish: 1406
Saved to: C:\Users\tammm\NQ and ES analysis\session_dataset.csv


---
## Cell 3 — Combinatorial Analysis

Loads `session_dataset.csv` and runs the full analysis for **both** targets (`ib_direction` and `ny_direction`):

- Individual features (all 31)
- 2-feature combinations (all pairs)
- 3-feature combinations (top N features only, to avoid combinatorial explosion)
- 4-feature combinations (top N features only)

Results are saved as CSVs prefixed with `ib_` or `ny_`.

In [3]:
import pandas as pd
import numpy as np
from itertools import combinations
import warnings
import os
warnings.filterwarnings('ignore')

# ── Load dataset ─────────────────────────────────────────────────────────────

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found — run Cell 2 first.\n{DATASET_PATH}")

session_df = pd.read_csv(DATASET_PATH)
session_df['date'] = pd.to_datetime(session_df['date'])
print(f"Loaded {len(session_df):,} trading days  "
      f"({session_df['date'].min().date()} → {session_df['date'].max().date()})")

# ── Feature list ─────────────────────────────────────────────────────────────
# Pre-9:30am features used for BOTH IB and NY session analysis.
# IB structure features (ib_close_half, ib_extreme_first) are only meaningful
# as predictors for the NY session — they are ignored when the target is ib_direction.

FEATURES_PRE_IB = [
    'prev_day_direction',
    'overnight_direction',
    'gap_type',
    'asia_direction',
    'london_direction',
    'globex_direction',
    'session_pattern',
    'asia_london_position',
    'globex_orb_direction',
    'london_orb_direction',
    'premarket_orb_direction',
    'ny_orb_direction',
    'ny_orb_first_break',
    'overnight_touched_prev_day_high',
    'overnight_touched_prev_day_low',
    'overnight_touched_prev_week_high',
    'overnight_touched_prev_week_low',
    'overnight_touched_r3',
    'overnight_touched_r2',
    'overnight_touched_r1',
    'overnight_touched_pp',
    'overnight_touched_gz',
    'overnight_touched_s1',
    'overnight_touched_s2',
    'overnight_touched_s3',
    'overnight_highest_pivot',
    'overnight_lowest_pivot',
    'ny_open_pivot_position',
    'ny_open_vs_prev_day',
    'day_of_week',
]

# IB structure features — only added when predicting the NY session
FEATURES_IB_STRUCTURE = [
    'ib_close_half',     # whether IB closed in upper or lower half of IB range
    'ib_extreme_first',  # whether IB high or low was printed first
]

# ── Core analysis helpers ─────────────────────────────────────────────────────

def quality_score(edge, n, max_n):
    """Weighted score: 60% edge, 40% log-scaled sample size."""
    edge_s   = edge / 50.0
    sample_s = np.log(n + 1) / np.log(max_n + 1)
    return EDGE_WEIGHT * edge_s + SAMPLE_WEIGHT * sample_s


def score_subset(subset, baseline, max_n, min_samples):
    """
    Given a filtered subset, compute metrics and return a dict or None
    if the subset doesn't meet quality thresholds.
    """
    n = len(subset)
    if n < min_samples:
        return None

    bullish_n = (subset['target'] == 'bullish').sum()
    bullish_p = bullish_n / n
    edge      = abs(bullish_p - 0.5) * 100

    if edge < MIN_EDGE_PCT:
        return None

    direction = 'bullish' if bullish_p > 0.5 else 'bearish'
    base      = baseline if direction == 'bullish' else (1 - baseline)
    conf_p    = bullish_p if direction == 'bullish' else (1 - bullish_p)
    lift      = conf_p / base

    return {
        'total_days'      : n,
        'bullish_count'   : int(bullish_n),
        'bearish_count'   : n - int(bullish_n),
        'bullish_pct'     : bullish_p * 100,
        'edge'            : edge,
        'lift'            : lift,
        'directional_bias': direction,
        'quality_score'   : quality_score(edge, n, max_n),
    }


def run_analysis(target_col, df, features, label):
    """
    Run 1/2/3/4-feature combination analysis for a given target column.
    Returns a dict of DataFrames keyed by combo size.
    """
    work = df[features + [target_col]].copy().rename(columns={target_col: 'target'})
    work = work[work['target'].notna()]

    total_days = len(work)
    bullish_n  = (work['target'] == 'bullish').sum()
    baseline   = bullish_n / total_days

    print(f"\n{'='*80}")
    print(f"TARGET: {label}  ({target_col})")
    print(f"{'='*80}")
    print(f"Days: {total_days}  |  Bullish: {bullish_n} ({baseline*100:.1f}%)  "
          f"Bearish: {total_days-bullish_n} ({(1-baseline)*100:.1f}%)")

    results = {}

    # ── 1-feature ────────────────────────────────────────────────────────────
    print(f"\n  Running individual feature analysis...")
    rows_1 = []
    feature_summary = []
    for feat in features:
        sub = work[work[feat].notna()]
        if len(sub) < 50:
            continue
        max_n = len(sub)
        for val in sub[feat].unique():
            s = score_subset(sub[sub[feat] == val], baseline, max_n, MIN_SAMPLES[1])
            if s:
                rows_1.append({'feature': feat, 'value': val, **s})

    df1 = pd.DataFrame(rows_1).sort_values('quality_score', ascending=False)
    results[1] = df1

    # Build per-feature summary for seeding 3/4-feature searches
    if len(df1):
        for feat, grp in df1.groupby('feature'):
            best = grp.loc[grp['quality_score'].idxmax()]
            feature_summary.append({'feature': feat, 'quality_score': best['quality_score']})
    feat_summary_df = (pd.DataFrame(feature_summary)
                       .sort_values('quality_score', ascending=False)
                       if feature_summary else pd.DataFrame(columns=['feature', 'quality_score']))

    print(f"  Found {len(df1)} individual values meeting filters.")

    # ── 2-feature ────────────────────────────────────────────────────────────
    print(f"  Running 2-feature combinations...")
    rows_2 = []
    for f1, f2 in combinations(features, 2):
        sub = work[[f1, f2, 'target']].dropna()
        if len(sub) < MIN_SAMPLES[2] * 2:
            continue
        max_n = len(sub)
        for (v1, v2), cnt in sub.groupby([f1, f2]).size().items():
            if cnt < MIN_SAMPLES[2]:
                continue
            s = score_subset(sub[(sub[f1]==v1) & (sub[f2]==v2)], baseline, max_n, MIN_SAMPLES[2])
            if s:
                rows_2.append({'features': f"{f1} + {f2}", 'values': f"{v1} + {v2}", **s})

    df2 = pd.DataFrame(rows_2).sort_values('quality_score', ascending=False)
    results[2] = df2
    print(f"  Found {len(df2)} 2-feature combos meeting filters.")

    # ── 3-feature ────────────────────────────────────────────────────────────
    top_feats = feat_summary_df.head(TOP_N_FOR_3FEATURE)['feature'].tolist()
    print(f"  Running 3-feature combinations (top {len(top_feats)} features)...")
    rows_3 = []
    for f1, f2, f3 in combinations(top_feats, 3):
        sub = work[[f1, f2, f3, 'target']].dropna()
        if len(sub) < MIN_SAMPLES[3] * 3:
            continue
        max_n = len(sub)
        for (v1, v2, v3), cnt in sub.groupby([f1, f2, f3]).size().items():
            if cnt < MIN_SAMPLES[3]:
                continue
            filt = (sub[f1]==v1) & (sub[f2]==v2) & (sub[f3]==v3)
            s = score_subset(sub[filt], baseline, max_n, MIN_SAMPLES[3])
            if s:
                rows_3.append({'features': f"{f1} + {f2} + {f3}",
                               'values'  : f"{v1} + {v2} + {v3}", **s})

    df3 = pd.DataFrame(rows_3).sort_values('quality_score', ascending=False)
    results[3] = df3
    print(f"  Found {len(df3)} 3-feature combos meeting filters.")

    # ── 4-feature ────────────────────────────────────────────────────────────
    top_feats4 = feat_summary_df.head(TOP_N_FOR_4FEATURE)['feature'].tolist()
    print(f"  Running 4-feature combinations (top {len(top_feats4)} features)...")
    rows_4 = []
    for f1, f2, f3, f4 in combinations(top_feats4, 4):
        sub = work[[f1, f2, f3, f4, 'target']].dropna()
        if len(sub) < MIN_SAMPLES[4] * 4:
            continue
        max_n = len(sub)
        for (v1, v2, v3, v4), cnt in sub.groupby([f1, f2, f3, f4]).size().items():
            if cnt < MIN_SAMPLES[4]:
                continue
            filt = (sub[f1]==v1) & (sub[f2]==v2) & (sub[f3]==v3) & (sub[f4]==v4)
            s = score_subset(sub[filt], baseline, max_n, MIN_SAMPLES[4])
            if s:
                rows_4.append({'features': f"{f1} + {f2} + {f3} + {f4}",
                               'values'  : f"{v1} + {v2} + {v3} + {v4}", **s})

    df4 = pd.DataFrame(rows_4).sort_values('quality_score', ascending=False)
    results[4] = df4
    print(f"  Found {len(df4)} 4-feature combos meeting filters.")

    return results, baseline


def save_results(results, prefix, baseline):
    """Save combo results to CSV, split by bullish/bearish."""
    saved = []
    for size, df_r in results.items():
        if df_r.empty:
            continue
        # Combined
        path = os.path.join(OUTPUT_DIR, f"{prefix}_{size}feature_combos.csv")
        df_r.to_csv(path, index=False)
        saved.append(path)
        # Top-100 split
        for bias in ['bullish', 'bearish']:
            sub = df_r[df_r['directional_bias'] == bias].head(100)
            p   = os.path.join(OUTPUT_DIR, f"{prefix}_{size}feature_top100_{bias.upper()}.csv")
            sub.to_csv(p, index=False)
            saved.append(p)
    print(f"\nSaved {len(saved)} CSV files for prefix '{prefix}'.")


# ── Run for IB direction — pre-IB features only ───────────────────────────

ib_results, ib_baseline = run_analysis('ib_direction', session_df, FEATURES_PRE_IB,
                                        'Initial Balance (9:30–10:30am)')
save_results(ib_results, 'ib', ib_baseline)

# ── Run for NY session direction — pre-IB features + IB structure ────────────

ny_results, ny_baseline = run_analysis('ny_direction', session_df,
                                        FEATURES_PRE_IB + FEATURES_IB_STRUCTURE,
                                        'NY Regular Session (9:30am–4:00pm)')
save_results(ny_results, 'ny', ny_baseline)

print("\n" + "="*80)
print("Cell 3 complete. Run Cell 4 to display results.")
print("="*80)

Loaded 3,109 trading days  (2013-01-02 → 2025-03-14)

TARGET: Initial Balance (9:30–10:30am)  (ib_direction)
Days: 3109  |  Bullish: 1627 (52.3%)  Bearish: 1482 (47.7%)

  Running individual feature analysis...
  Found 17 individual values meeting filters.
  Running 2-feature combinations...
  Found 896 2-feature combos meeting filters.
  Running 3-feature combinations (top 9 features)...
  Found 921 3-feature combos meeting filters.
  Running 4-feature combinations (top 8 features)...
  Found 1638 4-feature combos meeting filters.

Saved 12 CSV files for prefix 'ib'.

TARGET: NY Regular Session (9:30am–4:00pm)  (ny_direction)
Days: 3109  |  Bullish: 1703 (54.8%)  Bearish: 1406 (45.2%)

  Running individual feature analysis...
  Found 37 individual values meeting filters.
  Running 2-feature combinations...
  Found 1538 2-feature combos meeting filters.
  Running 3-feature combinations (top 10 features)...
  Found 764 3-feature combos meeting filters.
  Running 4-feature combinations (

---
## Cell 4 — Display Results

Loads the saved CSVs and displays:
1. Top patterns for the **Initial Balance** (IB)
2. Top patterns for the **NY Regular Session**

Patterns are shown at each complexity level (1–4 features) and filtered to a minimum confidence and sample size.

> Adjust `DISPLAY_MIN_SAMPLE` and `DISPLAY_MIN_CONFIDENCE` below to widen or tighten the filter without re-running the analysis.

In [4]:
import pandas as pd
import numpy as np
import os

# ── Display filter overrides (edit freely without re-running Cell 3) ──────────

DISPLAY_MIN_SAMPLE     = 50    # minimum days in subset
DISPLAY_MIN_CONFIDENCE = 65.0  # minimum directional confidence %
TOP_N_DISPLAY          = 50    # patterns to show per direction per complexity

# ── Helpers ───────────────────────────────────────────────────────────────────

def load_combo_files(prefix):
    """
    Load all combo CSVs for a given prefix (e.g. 'ib' or 'ny').
    Returns a combined DataFrame with a 'feature_count' column.
    """
    frames = []
    for size in [1, 2, 3, 4]:
        path = os.path.join(OUTPUT_DIR, f"{prefix}_{size}feature_combos.csv")
        if os.path.exists(path):
            df = pd.read_csv(path)
            df['feature_count'] = size
            # Normalise column names across 1-feature (feature/value) and combo (features/values)
            if 'feature' in df.columns and 'features' not in df.columns:
                df['features'] = df['feature']
                df['values']   = df['value'].astype(str)
            frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No combo files found for prefix '{prefix}'. Run Cell 3 first.")
    return pd.concat(frames, ignore_index=True)


def actionability_score(row, min_sample=DISPLAY_MIN_SAMPLE):
    edge   = row['edge']
    sample = row['total_days']
    conf_s = min(edge * 2, 100)
    if sample < min_sample:
        freq_s = (sample / min_sample) * 50
    else:
        freq_s = min(50 + (np.log2(sample / min_sample) * 25), 100)
    return round(conf_s * 0.6 + freq_s * 0.4, 1)


def display_session_results(prefix, session_label, baseline_bullish):
    """
    Print a formatted summary of top patterns for one session target.
    """
    baseline_bearish = 1 - baseline_bullish
    df = load_combo_files(prefix)

    # Compute confidence and true edge
    df['confidence'] = np.where(
        df['directional_bias'] == 'bullish',
        df['bullish_pct'],
        100 - df['bullish_pct']
    )
    df['true_edge'] = np.where(
        df['directional_bias'] == 'bullish',
        df['bullish_pct'] - (baseline_bullish * 100),
        (100 - df['bullish_pct']) - (baseline_bearish * 100)
    )
    df['action_score'] = df.apply(actionability_score, axis=1)

    # Apply display filters
    df = df[(df['total_days'] >= DISPLAY_MIN_SAMPLE) &
            (df['confidence'] >= DISPLAY_MIN_CONFIDENCE)]

    print("\n" + "="*130)
    print(f"  {session_label}")
    print(f"  Baseline: {baseline_bullish*100:.1f}% bullish | {baseline_bearish*100:.1f}% bearish")
    print(f"  Filter: ≥{DISPLAY_MIN_SAMPLE} days, ≥{DISPLAY_MIN_CONFIDENCE}% confidence")
    print("="*130)

    for size in [1, 2, 3, 4]:
        subset = df[df['feature_count'] == size].sort_values('action_score', ascending=False)
        if subset.empty:
            continue

        bull = subset[subset['directional_bias'] == 'bullish']
        bear = subset[subset['directional_bias'] == 'bearish']

        print(f"\n{'─'*130}")
        print(f"  {size}-FEATURE PATTERNS  "
              f"({len(subset):,} total | {len(bull):,} bullish | {len(bear):,} bearish)")
        print(f"{'─'*130}")

        for direction, grp in [('BULLISH', bull), ('BEARISH', bear)]:
            print(f"\n  TOP {min(TOP_N_DISPLAY, len(grp))} {direction} ({size}-feature):")
            if grp.empty:
                print("    No patterns meet filter criteria.")
                continue

            hdr = (f"  {'#':<3} {'Score':<7} {'Conf':<8} {'TrueEdge':<10} "
                   f"{'Days':<6} {'Lift':<6} {'Conditions'}")
            print(hdr)
            print("  " + "─" * 120)

            for i, (_, row) in enumerate(grp.head(TOP_N_DISPLAY).iterrows(), 1):
                cond = f"{row['features']}  =  {row['values']}" if size > 1 else \
                       f"{row['features']}  =  {row['values']}"
                cond = cond[:90]
                print(f"  {i:<3} {row['action_score']:<7.1f} "
                      f"{row['confidence']:<7.1f}% "
                      f"{row['true_edge']:>+7.1f}%   "
                      f"{int(row['total_days']):<6} "
                      f"{row['lift']:<6.2f} "
                      f"{cond}")

    # ── Summary statistics ────────────────────────────────────────────────────
    print(f"\n{'─'*130}")
    print(f"  SUMMARY — {session_label}")
    print(f"{'─'*130}")
    for size in [1, 2, 3, 4]:
        subset = df[df['feature_count'] == size]
        if subset.empty:
            continue
        bull = subset[subset['directional_bias'] == 'bullish']
        bear = subset[subset['directional_bias'] == 'bearish']
        print(f"  {size}-feature: {len(subset):4d} patterns  "
              f"| Bull {len(bull):4d} avg conf {bull['confidence'].mean():.1f}%  "
              f"avg {bull['total_days'].mean():.0f} days  "
              f"| Bear {len(bear):4d} avg conf {bear['confidence'].mean():.1f}%  "
              f"avg {bear['total_days'].mean():.0f} days"  if not bull.empty and not bear.empty
              else f"  {size}-feature: {len(subset):4d} patterns")

    # ── Export tradeable top-50 ───────────────────────────────────────────────
    for bias in ['bullish', 'bearish']:
        top = df[df['directional_bias'] == bias].sort_values('action_score', ascending=False).head(50)
        out = os.path.join(OUTPUT_DIR, f"tradeable_{prefix}_{bias}_top50.csv")
        top.to_csv(out, index=False)
    print(f"\n  Exported: tradeable_{prefix}_bullish_top50.csv / tradeable_{prefix}_bearish_top50.csv")


# ── Load baselines from saved dataset ────────────────────────────────────────

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError("session_dataset.csv not found. Run Cell 2 first.")

session_df = pd.read_csv(DATASET_PATH)
ib_baseline = (session_df['ib_direction'] == 'bullish').mean()
ny_baseline = (session_df['ny_direction'] == 'bullish').mean()

print(f"Baselines loaded from data:")
print(f"  IB  → {ib_baseline*100:.1f}% bullish")
print(f"  NY  → {ny_baseline*100:.1f}% bullish")

# ── Display both sessions ─────────────────────────────────────────────────────

display_session_results('ib', 'INITIAL BALANCE  (9:30–10:30am)', ib_baseline)
display_session_results('ny', 'NY REGULAR SESSION  (9:30am–4:00pm)', ny_baseline)

print("\n" + "="*130)
print("  Analysis complete.")
print("="*130)

Baselines loaded from data:
  IB  → 52.3% bullish
  NY  → 54.8% bullish

  INITIAL BALANCE  (9:30–10:30am)
  Baseline: 52.3% bullish | 47.7% bearish
  Filter: ≥50 days, ≥65.0% confidence

──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1-FEATURE PATTERNS  (3 total | 2 bullish | 1 bearish)
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

  TOP 2 BULLISH (1-feature):
  #   Score   Conf     TrueEdge   Days   Lift   Conditions
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1   63.3    69.4   %   +17.0%   2263   1.33   ny_orb_first_break  =  high
  2   62.1    68.4   %   +16.1%   1563   1.31   ny_orb_direction  =  bullish

  TOP 1 BEARISH (1-feature):
  #   Score   Conf     TrueEdge   Days   Lift   Conditions
  ──────────────────────────────────────